<a href="https://colab.research.google.com/github/newmanll/CMU_COMPBIO/blob/main/homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.2 MB/s eta 0:00:00


In [ ]:
import numpy as np
from IPython.display import HTML, display
import ipywidgets as widgets

# optional for protein matrices
from itertools import product
from Bio.Align import substitution_matrices

In [ ]:
DNA_ALPHABET = ["A","C","G","T"]

def simple_matrix(match=2, mismatch=-1):
    return {(a,b): (match if a==b else mismatch) for a in DNA_ALPHABET for b in DNA_ALPHABET}

# Load standard protein scoring matrices from Biopython
PROTEIN_MATRICES = {
    "BLOSUM62": substitution_matrices.load("BLOSUM62"),
    "PAM250": substitution_matrices.load("PAM250"),
    "BLOSUM50": substitution_matrices.load("BLOSUM50")
}

def get_score(a,b,mode="DNA", dna_match=2, dna_mismatch=-1, protein_scoring_matrix_name="BLOSUM62"):
    if a=="-" or b=="-":
        return -2 # Gap extension penalty is handled separately in alignment functions
    if mode=="DNA":
        return dna_match if a==b else dna_mismatch
    else: # Protein mode
        matrix = PROTEIN_MATRICES.get(protein_scoring_matrix_name, PROTEIN_MATRICES["BLOSUM62"]) # Default to BLOSUM62 if not found
        # Biopython's substitution matrices can be indexed directly to get the score
        return matrix[a, b]

In [ ]:
GENETIC_CODE = {
    'ATA':'I', 'ATC':'I', 'ATT':'I', 'ATG':'M',
    'ACA':'T', 'ACC':'T', 'ACG':'T', 'ACT':'T',
    'AAC':'N', 'AAT':'N', 'AAA':'K', 'AAG':'K',
    'AGC':'S', 'AGT':'S', 'AGA':'R', 'AGG':'R',
    'CTA':'L', 'CTC':'L', 'CTG':'L', 'CTT':'L',
    'CCA':'P', 'CCC':'P', 'CCG':'P', 'CCT':'P',
    'CAC':'H', 'CAT':'H', 'CAA':'Q', 'CAG':'Q',
    'CGA':'R', 'CGC':'R', 'CGG':'R', 'CGT':'R',
    'GTA':'V', 'GTC':'V', 'GTG':'V', 'GTT':'V',
    'GCA':'A', 'GCC':'A', 'GCG':'A', 'GCT':'A',
    'GAC':'D', 'GAT':'D', 'GAA':'E', 'GAG':'E',
    'GGA':'G', 'GGC':'G', 'GGG':'G', 'GGT':'G',
    'TCA':'S', 'TCC':'S', 'TCG':'S', 'TCT':'S',
    'TTC':'F', 'TTT':'F', 'TTA':'L', 'TTG':'L',
    'TAC':'Y', 'TAT':'Y', 'TAA':'*', 'TAG':'*', # '*' denotes stop codon
    'TGC':'C', 'TGT':'C', 'TGA':'*', 'TGG':'W',
}

In [ ]:
def translate_aligned_dna_to_protein(aligned_dna_seq):
    # Remove gaps for translation, then reinsert later based on protein alignment
    ungapped_dna = aligned_dna_seq.replace('-', '')
    protein_seq = ''
    for i in range(0, len(ungapped_dna) - len(ungapped_dna) % 3, 3):
        codon = ungapped_dna[i:i+3]
        if codon in GENETIC_CODE:
            protein_seq += GENETIC_CODE[codon]
        else:
            # Handle unknown codons (e.g., if sequence is truncated or has errors)
            protein_seq += 'X'
    return protein_seq


In [ ]:
def analyze_alignment_discrepancies(aligned_dna1, aligned_dna2, aligned_protein1, aligned_protein2):
    if len(aligned_protein1) != len(aligned_protein2):
        raise ValueError("Aligned protein sequences must have the same length.")

    # Convert aligned DNA to its conceptual protein sequence *with respect to its own gaps*
    # This is slightly different from `translate_aligned_dna_to_protein`
    # We need to map DNA alignment positions to protein alignment positions

    # Helper to create a mapping from protein index to DNA triplet start index
    def get_protein_to_dna_map(aligned_dna):
        dna_idx = 0
        protein_map = {}
        for p_idx in range(len(aligned_protein1)): # Iterate based on the length of the protein alignment
            # Find the starting DNA index for the codon that maps to this protein position
            while dna_idx < len(aligned_dna) and aligned_dna[dna_idx] == '-':
                dna_idx += 1 # Skip gaps in DNA
            if dna_idx < len(aligned_dna):
                protein_map[p_idx] = dna_idx # Store the start of the codon
                # Advance dna_idx by 3, accounting for gaps in the next 3 positions
                for _ in range(3):
                    if dna_idx < len(aligned_dna):
                        dna_idx += 1
                        while dna_idx < len(aligned_dna) and aligned_dna[dna_idx] == '-':
                            dna_idx += 1
        return protein_map

    # Get mappings for both DNA sequences
    dna1_to_p_map = get_protein_to_dna_map(aligned_dna1)
    dna2_to_p_map = get_protein_to_dna_map(aligned_dna2)

    synonymous_mutations = []
    non_synonymous_mutations = []
    dna_only_changes = [] # Where DNA differs, but protein sequence is not considered (e.g. gaps in protein)

    for p_idx in range(len(aligned_protein1)):
        aa1 = aligned_protein1[p_idx]
        aa2 = aligned_protein2[p_idx]

        # Get the corresponding DNA codons/regions
        codon1_start_idx = dna1_to_p_map.get(p_idx)
        codon2_start_idx = dna2_to_p_map.get(p_idx)

        if codon1_start_idx is None or codon2_start_idx is None: # Likely a gap in protein alignment
            # If protein has a gap, DNA difference here isn't a 'synonymous' or 'non-synonymous' mutation in the typical sense
            continue

        # Extract codons, handling gaps within the codon region from the aligned DNA
        codon1_raw = aligned_dna1[codon1_start_idx:codon1_start_idx+3]
        codon2_raw = aligned_dna2[codon2_start_idx:codon2_start_idx+3]

        # Filter out gaps for actual codon comparison
        codon1 = codon1_raw.replace('-', '')
        codon2 = codon2_raw.replace('-', '')

        # Only consider positions where both are valid codons (length 3) for translation
        if len(codon1) == 3 and len(codon2) == 3:
            dna_differs = (codon1 != codon2)
            protein_same = (aa1 == aa2 and aa1 != '-')
            protein_differs = (aa1 != aa2 and aa1 != '-' and aa2 != '-')

            if dna_differs and protein_same:
                synonymous_mutations.append({
                    'protein_position': p_idx,
                    'aa1': aa1, 'aa2': aa2,
                    'dna1_codon': codon1, 'dna2_codon': codon2
                })
            elif dna_differs and protein_differs:
                non_synonymous_mutations.append({
                    'protein_position': p_idx,
                    'aa1': aa1, 'aa2': aa2,
                    'dna1_codon': codon1, 'dna2_codon': codon2
                })
        elif codon1_raw != codon2_raw: # DNA differs, but one or both are not complete codons (e.g., due to indels)
             dna_only_changes.append({
                    'protein_position': p_idx,
                    'dna1_segment': codon1_raw, 'dna2_segment': codon2_raw,
                    'protein_segment1': aa1, 'protein_segment2': aa2
                })

    return {
        'synonymous_mutations': synonymous_mutations,
        'non_synonymous_mutations': non_synonymous_mutations,
        'dna_gaps_not_full_codons': dna_only_changes
    }


In [ ]:
def needleman_wunsch(s1, s2, score_fn, gap_penalty=-2):
    n, m = len(s1), len(s2)
    dp = np.zeros((n+1, m+1))
    trace = np.zeros((n+1, m+1), dtype=int)

    for i in range(1, n+1):
        dp[i][0] = i * gap_penalty
        trace[i][0] = 1 # 1 for deletion (from s1)
    for j in range(1, m+1):
        dp[0][j] = j * gap_penalty
        trace[0][j] = 2 # 2 for insertion (from s2)

    for i in range(1, n+1):
        for j in range(1, m+1):
            match = dp[i-1][j-1] + score_fn(s1[i-1], s2[j-1])
            delete = dp[i-1][j] + gap_penalty
            insert = dp[i][j-1] + gap_penalty
            dp[i][j] = max(match, delete, insert)
            trace[i][j] = np.argmax([match, delete, insert]) # 0:match, 1:delete, 2:insert

    # traceback
    i, j = n, m
    a1, a2 = [], []
    while i>0 or j>0:
        if i>0 and j>0 and trace[i][j]==0: # Match/Mismatch
            a1.append(s1[i-1]); a2.append(s2[j-1])
            i-=1; j-=1
        elif i>0 and (j==0 or trace[i][j]==1): # Deletion in s2 (gap in a2)
            a1.append(s1[i-1]); a2.append("-")
            i-=1
        else: # Insertion in s2 (gap in a1)
            a1.append("-"); a2.append(s2[j-1])
            j-=1

    return dp[n][m], "".join(reversed(a1)), "".join(reversed(a2))

In [ ]:
def smith_waterman(s1, s2, score_fn, gap_penalty=-2):
    n, m = len(s1), len(s2)
    dp = np.zeros((n+1, m+1))

    max_score = 0
    max_i = max_j = 0

    for i in range(1,n+1):
        for j in range(1,m+1):
            match = dp[i-1][j-1] + score_fn(s1[i-1], s2[j-1])
            delete = dp[i-1][j] + gap_penalty
            insert = dp[i][j-1] + gap_penalty
            dp[i][j] = max(0, match, delete, insert) # Local alignment: score cannot be negative

            if dp[i][j] > max_score:
                max_score = dp[i][j]
                max_i, max_j = i, j

    # traceback
    i, j = max_i, max_j
    a1, a2 = [], []

    while i>0 and j>0 and dp[i][j] != 0: # Stop when score becomes 0
        # Determine which path led to dp[i][j]
        # Prioritize match, then delete, then insert, if scores are equal
        if dp[i][j] == dp[i-1][j-1] + score_fn(s1[i-1], s2[j-1]):
            a1.append(s1[i-1]); a2.append(s2[j-1])
            i-=1; j-=1
        elif dp[i][j] == dp[i-1][j] + gap_penalty:
            a1.append(s1[i-1]); a2.append("-")
            i-=1
        elif dp[i][j] == dp[i][j-1] + gap_penalty:
            a1.append("-"); a2.append(s2[j-1])
            j-=1
        else:
            # This case should ideally not be reached if logic is correct and scores match
            # But as a fallback, if no path matches, break to prevent infinite loop
            break

    return max_score, "".join(reversed(a1)), "".join(reversed(a2))

In [ ]:
def alignment_stats(a1, a2, mode='DNA', protein_scoring_matrix_name='BLOSUM62'):
    matches = 0
    aligned_positions = 0 # Positions where neither is a gap

    for x, y in zip(a1, a2):
        if x != '-' and y != '-': # Consider only non-gap positions for identity/similarity calculation base
            aligned_positions += 1
            if x == y:
                matches += 1

    identity = (matches / aligned_positions * 100) if aligned_positions else 0

    total_gaps = a1.count('-') + a2.count('-')

    # --- New gap analysis for biological plausibility ---
    num_gap_blocks = 0
    total_gap_lengths_sum = 0 # Sum of lengths of all contiguous gap blocks
    frameshift_indel_count = 0 # Count of gap blocks that are not multiples of 3 (for DNA)

    def get_gap_blocks_info(sequence, current_mode):
        nonlocal num_gap_blocks, total_gap_lengths_sum, frameshift_indel_count
        in_gap = False
        current_gap_len = 0
        for char in sequence:
            if char == '-':
                if not in_gap:
                    in_gap = True
                    num_gap_blocks += 1
                current_gap_len += 1
            else:
                if in_gap:
                    total_gap_lengths_sum += current_gap_len
                    if current_mode == 'DNA' and current_gap_len % 3 != 0:
                        frameshift_indel_count += 1
                    in_gap = False
                    current_gap_len = 0
        # Check for a trailing gap
        if in_gap:
            total_gap_lengths_sum += current_gap_len
            if current_mode == 'DNA' and current_gap_len % 3 != 0:
                frameshift_indel_count += 1

    get_gap_blocks_info(a1, mode)
    get_gap_blocks_info(a2, mode)

    avg_gap_block_length = round(total_gap_lengths_sum / num_gap_blocks, 2) if num_gap_blocks else 0

    return {
        "identity": round(identity, 2),
        "total_gaps": total_gaps,
        "num_gap_blocks": num_gap_blocks,
        "avg_gap_block_length": avg_gap_block_length,
        "frameshift_indel_count": frameshift_indel_count if mode == 'DNA' else 'N/A' # N/A for protein
    }

In [ ]:
def color_align(a1, a2):
    html = "<pre style='font-size:16px'>"
    for x,y in zip(a1,a2):
        if x==y:
            c="green"
        elif x=="-" or y=="-":
            c="gray"
        else:
            c="red"
        html += f"<span style='color:{c}'>{x}</span> "
    html += "<br>"
    for x,y in zip(a1,a2):
        if x==y:
            c="green"
        elif x=="-" or y=="-":
            c="gray"
        else:
            c="red"
        html += f"<span style='color:{c}'>{y}</span> "
    html += "</pre>"
    display(HTML(html))

In [ ]:
# Define default sequences for SARS-CoV and SARS-CoV-2 spike protein gene/protein
# These are placeholder sequences. For actual analysis, you should fetch them from NCBI.
SARS_COV_DNA_DEFAULT = "ATGTTCATTTTCTTTTATTCTTTTTCTTTTTTCTTTTTTCTTTTTAAAAGATCCAGTATGTGGAACTACTCAACCTGAATGATCTTTGAAATGTTAAGAAATCAATCAAGTGTAGTATCTCGTCATTATGAAGAGACTTTGTTGGGTCAATTTTGCTTCAAGTGAAAAGAATTTGTACACTAAAAAACTTCCTCTGAAACAACAAAAAGAATTTGGAAAATGATTGCTGTGCGTGTTTTACAGAAAGGCAGACAAGCTGTTTCAACGAATGTGATAAAAACCCGTATATTATATGGGTTAAGGTATGTATTTGAATGCGAAAATCAAAATTGTAATCCAAATATTGTTGGTGAACGAATATCTACAGAAAAGGCTAGTATTTATGAACAGTTATGGAGTTGCGGTTGCCATTTTGTTGAAATCACCTTCGTGATAAGAGTCGTTTTGCACCGGTACATGTGTTGGAACTTATTTAAATTCACAAGATGGTGTTCATTGTTGATGCCATTCTTTGGTAAAGGACTAGTTGTGCGTGTTTTAACAGAAAGGCAGACAAGCTGTTTCAACGAATGTGATAAAAACCCGTATATTATATGGGTTAAGGTATGTATTTGAATGCGAAAATCAAAATTGTAATCCAAATATTGTTGGTGAACGAATATCTACAGAAAAGGCTAGTATTTATGAACAGTTATGGAGTTGCGGTTGCCATTTTGTTGGAAATCACCTTCGTGATAAGAGTCGATTGCACCGGTACATGTGTTGGAACTTATTTAAATTCACAAGATGGTGTTCATTGTTGATGCCATTCTTTGGTAAAGGACTAGTTGTGCGT"
SARS_COV2_DNA_DEFAULT = "ATGTTTGTTTTTCTTGTTTTATTGCCACTAGTCTCTAGTCAGTGTGTTAATCTTACAACCAGAACTCAATTACCCCCTGCATACACTAATTCTTTCACACGTGGTTCATTTGGCAGTTGTTTTTTAATAATTGTGTATCGTTGATTAGTGTTTCACCGGGGCATTTGATCCATCAACAAGGCCACCTACAAATGTGTCAATCAGTGTGGTTTACATAACTAATAACTACATTCAGGAGACCAATTACCCAATGGTAATTTTTCATGCACATCAGCACAAACAAAACTTATTTAATAATAAATTACCTAGAGATATATTCACCCGGTTTATTGCCTGCGAAGGCGGGTATGGCGTTAATGTATTATCTTTCTCATACACCAATCAGTTTCCATGGAAACAAAGCCTTGTTTATTCTACTAATTTGGTGTTAAAAATGCGCGTTTCTGTTGCAGGCTGGAGTGCAATCATCGAACGTTGGGAATGTAATATTCATCTTGTTTTGCTTTCAGTAAGTCACGACGGTAAGTTGCATGTGAATTCACAAAACTTTAGTGAAACTTGAGTATTTAATGTACACTAAAACAAACTTCATTTACCGGTTTATGAGCTCGTAATGGATACTTCTGTGTGCCAAATTAGGACCACTTTATTACAGATGCATTAAACAACTTTGTAGAATTATTATGGGATATAAACTTGTAGTGTGGAATTTCACAGACGGTGTTTTACAGAAAGGCAGACAAGCTGTTTGTCAACGAATGTGATAAAAACCCGTATATTATATGGGTTAAGGTATGTATTTGAATGCGAAAATCAAAATTGTAATCCAAATATTGTTGGTGAACGAATATCTACAGAAAAGGCTAGTATTTATGAACAGTTATGGAGTTGCGGTTGCCATTTTGTTGAAATCACCTTCGTGATAAGAGTCGTTTTGCACCGGTACATGTGTTGGAACTTATTTAAATTCACAAGATGGTGTTCATTGTTGATGCCATTCTTTGGTAAAGGACTAGTTGTGCGTGTTTTAACAGAAAGGCAGACAAGCTGTTTGTCAACGAATGTGATAAAAACCCGTATATTATATGGGTTAAGGTATGTATTTGAATGCGAAAATCAAAATTGTAATCCAAATATTGTTGGTGAACGAATATCTACAGAAAAGGCTAGTATTTATGAACAGTTATGGAGTTGCGGTTGCCATTTTGTTGGAAATCACCTTCGTGATAAGAGTCGTTTTGCACCGGTACATGTGTTGGAACTTATTTAAATTCACAAGATGGTGTTCATTGTTGATGCCATTCTTTGGTAAAGGACTAGTTGTGCGT"

SARS_COV_PROTEIN_DEFAULT = "MFLFLLSLTSTGSEVGKVNGLTSVSLPSGFQPLVDYQTFTSDYLQPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATVSIDIQAYEQAVKIDDSRFEVLADHKKQLTPLLREVGELKNVVLPLVGTALLTLTFNYNAEDFLPFYQHLVFNKFLSFHSGTVRLPNLQSMNYNPNFTGHILRQSSETRIDLLFLKVNDRVSDIDLFCKFLTLRNIEFVLADHKKQLTPLLREVGELKNVVLPLVGTALLTLTFNYNAEDFLPFYQHLVFNKFLSFHSGTVRLPNLQSMNYNPNFTGHILRQSSETRIDLLFLKVNDR"
SARS_COV2_PROTEIN_DEFAULT = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATVSIDLFCKFLTLRNIEFVLADHKKQLTPLLREVGELKNVVLPLVGTALLTLTFNYNAEDFLPFYQHLVFNKFLSFHSGTVRLPNLQSMNYNPNFTGHILRQSSETRIDLLFLKVNDRVSIDLFCKFLTLRNIEFVLADHKKQLTPLLREVGELKNVVLPLVGTALLTLTFNYNAEDFLPFYQHLVFNKFLSFHSGTVRLPNLQSMNYNPNFTGHILRQSSETRIDLLFLKVNDRVSIDLFCKFLTLRNIEFVLADHKKQLTPLLREVGELKNVVLPLVGTALLTLTFNYNAEDFLPFYQHLVFNKFLSFHSGTVRLPNLQSMNYNPNFTGHILRQSSETRIDLLFLKVNDRV"

### Demonstration of DNA and Protein Alignment Discrepancy Analysis

Here, we will:
1. Perform a global DNA alignment between SARS-CoV and SARS-CoV-2 spike DNA sequences.
2. Translate the *ungapped* protein sequences from the original DNA (or use provided protein sequences if available).
3. Perform a global protein alignment between the two protein sequences.
4. Use the `analyze_alignment_discrepancies` function to compare the aligned DNA and protein results.

In [ ]:
# 1. Get original sequences
dna_sars_cov = SARS_COV_DNA_DEFAULT
dna_sars_cov2 = SARS_COV2_DNA_DEFAULT

protein_sars_cov = SARS_COV_PROTEIN_DEFAULT
protein_sars_cov2 = SARS_COV2_PROTEIN_DEFAULT

print("--- Performing Global DNA Alignment ---")
dna_score, dna_aligned1, dna_aligned2 = needleman_wunsch(
    dna_sars_cov,
    dna_sars_cov2,
    lambda c1, c2: get_score(c1, c2, mode='DNA', dna_match=2, dna_mismatch=-1),
    gap_penalty=-2
)

print(f"DNA Alignment Score: {dna_score}")
print("Aligned DNA Sequences:")
color_align(dna_aligned1, dna_aligned2)
dna_stats = alignment_stats(dna_aligned1, dna_aligned2)
print(f"DNA Identity: {dna_stats['identity']}% | Total Gaps: {dna_stats['total_gaps']} | Gap Blocks: {dna_stats['num_gap_blocks']} | Avg Gap Length: {dna_stats['avg_gap_block_length']} | Frameshift Indels: {dna_stats['frameshift_indel_count']}")

print("\n--- Performing Global Protein Alignment ---")
protein_score, protein_aligned1, protein_aligned2 = needleman_wunsch(
    protein_sars_cov,
    protein_sars_cov2,
    lambda c1, c2: get_score(c1, c2, mode='Protein', protein_scoring_matrix_name='BLOSUM62'),
    gap_penalty=-10 # Protein gap penalties are usually higher
)

print(f"Protein Alignment Score: {protein_score}")
print("Aligned Protein Sequences:")
color_align(protein_aligned1, protein_aligned2)
protein_stats = alignment_stats(protein_aligned1, protein_aligned2)
print(f"Protein Identity: {protein_stats['identity']}% | Total Gaps: {protein_stats['total_gaps']} | Gap Blocks: {protein_stats['num_gap_blocks']} | Avg Gap Length: {protein_stats['avg_gap_block_length']}")

print("\n--- Analyzing Discrepancies between DNA and Protein Alignments ---")
discrepancy_results = analyze_alignment_discrepancies(
    dna_aligned1,
    dna_aligned2,
    protein_aligned1,
    protein_aligned2
)

print("\nSynonymous Mutations (DNA change, Protein same):")
if discrepancy_results['synonymous_mutations']:
    for m in discrepancy_results['synonymous_mutations']:
        print(f"  Protein Pos {m['protein_position']}: {m['dna1_codon']} -> {m['dna2_codon']} (AA: {m['aa1']} -> {m['aa2']})")
else:
    print("  None found in this alignment.")

print("\nNon-Synonymous Mutations (DNA change, Protein change):")
if discrepancy_results['non_synonymous_mutations']:
    for m in discrepancy_results['non_synonymous_mutations']:
        print(f"  Protein Pos {m['protein_position']}: {m['dna1_codon']} -> {m['dna2_codon']} (AA: {m['aa1']} -> {m['aa2']})")
else:
    print("  None found in this alignment.")

print("\nDNA segment differences where translation was unclear (e.g., gaps or incomplete codons):")
if discrepancy_results['dna_gaps_not_full_codons']:
    for m in discrepancy_results['dna_gaps_not_full_codons']:
        print(f"  Protein Pos {m['protein_position']}: DNA {m['dna1_segment']} vs {m['dna2_segment']} (Protein: {m['protein_segment1']} vs {m['protein_segment2']})")
else:
    print("  None found in this alignment.")

print("\n--- Interpretation ---")
print("The differences in DNA alignment that result in the same amino acid in the protein alignment are synonymous mutations. These mutations do not change the protein sequence and are often considered 'silent'.")
print("Differences in DNA that lead to different amino acids are non-synonymous mutations. These changes alter the protein and can have functional consequences.")
print("Differences in DNA segments that couldn't be clearly translated to protein (often due to alignment gaps leading to incomplete codons) indicate insertions or deletions at the DNA level that affect the reading frame or length, which are highly significant at the protein level.")

--- Performing Global DNA Alignment ---
DNA Alignment Score: 672.0
Aligned DNA Sequences:


DNA Identity: 99.52% | Total Gaps: 493 | Gap Blocks: 152 | Avg Gap Length: 3.24 | Frameshift Indels: 119

--- Performing Global Protein Alignment ---
Protein Alignment Score: 447.0
Aligned Protein Sequences:


Protein Identity: 71.99% | Total Gaps: 95 | Gap Blocks: 59 | Avg Gap Length: 1.61

--- Analyzing Discrepancies between DNA and Protein Alignments ---

Synonymous Mutations (DNA change, Protein same):
  Protein Pos 12: TTT -> AGT (AA: S -> S)
  Protein Pos 47: TTG -> TGA (AA: L -> L)
  Protein Pos 59: CAC -> AAG (AA: S -> S)
  Protein Pos 81: AAA -> TTA (AA: P -> P)
  Protein Pos 84: CAA -> GGT (AA: P -> P)
  Protein Pos 90: GTG -> TCA (AA: Y -> Y)
  Protein Pos 94: CGT -> AAA (AA: T -> T)
  Protein Pos 97: TAT -> TAA (AA: S -> S)
  Protein Pos 101: TAT -> ACC (AA: R -> R)
  Protein Pos 102: GTA -> TAG (AA: G -> G)
  Protein Pos 110: TGT -> TGC (AA: D -> D)
  Protein Pos 115: GTT -> GTA (AA: S -> S)
  Protein Pos 123: AAG -> CTC (AA: T -> T)
  Protein Pos 127: TAT -> TCA (AA: D -> D)
  Protein Pos 128: GAA -> GTT (AA: L -> L)
  Protein Pos 129: CAG -> TCC (AA: F -> F)
  Protein Pos 132: AGT -> ACA (AA: F -> F)
  Protein Pos 137: TTT -> TTC (AA: N -> N)
  Protein Pos 143: GTG -> AAA (AA:

In [ ]:
def needleman_wunsch(s1, s2, score_fn, gap_penalty=-2):
    n, m = len(s1), len(s2)
    dp = np.zeros((n+1, m+1))
    trace = np.zeros((n+1, m+1), dtype=int)

    for i in range(1, n+1):
        dp[i][0] = i * gap_penalty
        trace[i][0] = 1 # 1 for deletion (from s1)
    for j in range(1, m+1):
        dp[0][j] = j * gap_penalty
        trace[0][j] = 2 # 2 for insertion (from s2)

    for i in range(1, n+1):
        for j in range(1, m+1):
            match = dp[i-1][j-1] + score_fn(s1[i-1], s2[j-1])
            delete = dp[i-1][j] + gap_penalty
            insert = dp[i][j-1] + gap_penalty
            dp[i][j] = max(match, delete, insert)
            trace[i][j] = np.argmax([match, delete, insert]) # 0:match, 1:delete, 2:insert

    # traceback
    i, j = n, m
    a1, a2 = [], []
    while i>0 or j>0:
        if i>0 and j>0 and trace[i][j]==0: # Match/Mismatch
            a1.append(s1[i-1]); a2.append(s2[j-1])
            i-=1; j-=1
        elif i>0 and (j==0 or trace[i][j]==1): # Deletion in s2 (gap in a2)
            a1.append(s1[i-1]); a2.append("-")
            i-=1
        else: # Insertion in s2 (gap in a1)
            a1.append("-"); a2.append(s2[j-1])
            j-=1

    return dp[n][m], "".join(reversed(a1)), "".join(reversed(a2))


In [ ]:
def smith_waterman(s1, s2, score_fn, gap_penalty=-2):
    n, m = len(s1), len(s2)
    dp = np.zeros((n+1, m+1))

    max_score = 0
    max_i = max_j = 0

    for i in range(1,n+1):
        for j in range(1,m+1):
            match = dp[i-1][j-1] + score_fn(s1[i-1], s2[j-1])
            delete = dp[i-1][j] + gap_penalty
            insert = dp[i][j-1] + gap_penalty
            dp[i][j] = max(0, match, delete, insert) # Local alignment: score cannot be negative

            if dp[i][j] > max_score:
                max_score = dp[i][j]
                max_i, max_j = i, j

    # traceback
    i, j = max_i, max_j
    a1, a2 = [], []

    while i>0 and j>0 and dp[i][j] != 0: # Stop when score becomes 0
        # Determine which path led to dp[i][j]
        # Prioritize match, then delete, then insert, if scores are equal
        if dp[i][j] == dp[i-1][j-1] + score_fn(s1[i-1], s2[j-1]):
            a1.append(s1[i-1]); a2.append(s2[j-1])
            i-=1; j-=1
        elif dp[i][j] == dp[i-1][j] + gap_penalty:
            a1.append(s1[i-1]); a2.append("-")
            i-=1
        elif dp[i][j] == dp[i][j-1] + gap_penalty:
            a1.append("-"); a2.append(s2[j-1])
            j-=1
        else:
            # This case should ideally not be reached if logic is correct and scores match
            # But as a fallback, if no path matches, break to prevent infinite loop
            break

    return max_score, "".join(reversed(a1)), "".join(reversed(a2))


In [ ]:
def alignment_stats(a1, a2, mode='DNA', protein_scoring_matrix_name='BLOSUM62'):
    matches = 0
    aligned_positions = 0 # Positions where neither is a gap

    for x, y in zip(a1, a2):
        if x != '-' and y != '-': # Consider only non-gap positions for identity/similarity calculation base
            aligned_positions += 1
            if x == y:
                matches += 1

    identity = (matches / aligned_positions * 100) if aligned_positions else 0

    total_gaps = a1.count('-') + a2.count('-')

    # --- New gap analysis for biological plausibility ---
    num_gap_blocks = 0
    total_gap_lengths_sum = 0 # Sum of lengths of all contiguous gap blocks
    frameshift_indel_count = 0 # Count of gap blocks that are not multiples of 3 (for DNA)

    def get_gap_blocks_info(sequence, current_mode):
        nonlocal num_gap_blocks, total_gap_lengths_sum, frameshift_indel_count
        in_gap = False
        current_gap_len = 0
        for char in sequence:
            if char == '-':
                if not in_gap:
                    in_gap = True
                    num_gap_blocks += 1
                current_gap_len += 1
            else:
                if in_gap:
                    total_gap_lengths_sum += current_gap_len
                    if current_mode == 'DNA' and current_gap_len % 3 != 0:
                        frameshift_indel_count += 1
                    in_gap = False
                    current_gap_len = 0
        # Check for a trailing gap
        if in_gap:
            total_gap_lengths_sum += current_gap_len
            if current_mode == 'DNA' and current_gap_len % 3 != 0:
                frameshift_indel_count += 1

    get_gap_blocks_info(a1, mode)
    get_gap_blocks_info(a2, mode)

    avg_gap_block_length = round(total_gap_lengths_sum / num_gap_blocks, 2) if num_gap_blocks else 0

    return {
        "identity": round(identity, 2),
        "total_gaps": total_gaps,
        "num_gap_blocks": num_gap_blocks,
        "avg_gap_block_length": avg_gap_block_length,
        "frameshift_indel_count": frameshift_indel_count if mode == 'DNA' else 'N/A' # N/A for protein
    }

In [ ]:
def color_align(a1, a2):
    html = "<pre style='font-size:16px'>"
    for x,y in zip(a1,a2):
        if x==y:
            c="green"
        elif x=="-" or y=="-":
            c="gray"
        else:
            c="red"
        html += f"<span style='color:{c}'>{x}</span> "
    html += "<br>"
    for x,y in zip(a1,a2):
        if x==y:
            c="green"
        elif x=="-" or y=="-":
            c="gray"
        else:
            c="red"
        html += f"<span style='color:{c}'>{y}</span> "
    html += "</pre>"
    display(HTML(html))

### Interactive Alignment Widget

Use the widgets below to input your sequences and select the alignment parameters. Click 'Run Alignment' to see the results.

In [ ]:
import numpy as np
from IPython.display import HTML, display
import ipywidgets as widgets

# optional for protein matrices
from itertools import product

# Define default sequences for SARS-CoV and SARS-CoV-2 spike protein gene/protein
# These are placeholder sequences. For actual analysis, you should fetch them from NCBI.
SARS_COV_DNA_DEFAULT = "ATGTTCATTTTCTTTTATTCTTTTTCTTTTTTCTTTTTTCTTTTTAAAAGATCCAGTATGTGGAACTACTCAACCTGAATGATCTTTGAAATGTTAAGAAATCAATCAAGTGTAGTATCTCGTCATTATGAAGAGACTTTGTTGGGTCAATTTTGCTTCAAGTGAAAAGAATTTGTACACTAAAAAACTTCCTCTGAAACAACAAAAAGAATTTGGAAAATGATTGCTGTGCGTGTTTTACAGAAAGGCAGACAAGCTGTTTCAACGAATGTGATAAAAACCCGTATATTATATGGGTTAAGGTATGTATTTGAATGCGAAAATCAAAATTGTAATCCAAATATTGTTGGTGAACGAATATCTACAGAAAAGGCTAGTATTTATGAACAGTTATGGAGTTGCGGTTGCCATTTTGTTGAAATCACCTTCGTGATAAGAGTCGTTTTGCACCGGTACATGTGTTGGAACTTATTTAAATTCACAAGATGGTGTTCATTGTTGATGCCATTCTTTGGTAAAGGACTAGTTGTGCGTGTTTTAACAGAAAGGCAGACAAGCTGTTTCAACGAATGTGATAAAAACCCGTATATTATATGGGTTAAGGTATGTATTTGAATGCGAAAATCAAAATTGTAATCCAAATATTGTTGGTGAACGAATATCTACAGAAAAGGCTAGTATTTATGAACAGTTATGGAGTTGCGGTTGCCATTTTGTTGGAAATCACCTTCGTGATAAGAGTCRATTGCACCGGTACATGTGTTGGAACTTATTTAAATTCACAAGATGGTGTTCATTGTTGATGCCATTCTTTGGTAAAGGACTAGTTGTGCGT"
SARS_COV2_DNA_DEFAULT = "ATGTTTGTTTTTCTTGTTTTATTGCCACTAGTCTCTAGTCAGTGTGTTAATCTTACAACCAGAACTCAATTACCCCCTGCATACACTAATTCTTTCACACGTGGTTCATTTGGCAGTTGTTTTTTAATAATTGTGTATCGTTGATTAGTGTTTCACCGGGGCATTTGATCCATCAACAAGGCCACCTACAAATGTGTCAATCAGTGTGGTTTACATAACTAATAACTACATTCAGGAGACCAATTACCCAATGGTAATTTTTCATGCACATCAGCACAAACAAAACTTATTTAATAATAAATTACCTAGAGATATATTCACCCGGTTTATTGCCTGCGAAGGCGGGTATGGCGTTAATGTATTATCTTTCTCATACACCAATCAGTTTCCATGGAAACAAAGCCTTGTTTATTCTACTAATTTGGTGTTAAAAATGCGCGTTTCTGTTGCAGGCTGGAGTGCAATCATCGAACGTTGGGAATGTAATATTCATCTTGTTTTGCTTTCAGTAAGTCACGACGGTAAGTTGCATGTGAATTCACAAAACTTTAGTGAAACTTGAGTATTTAATGTACACTAAAACAAACTTCATTTACCGGTTTATGAGCTCGTAATGGATACTTCTGTGTGCCAAATTAGGACCACTTTATTACAGATGCATTAAACAACTTTGTAGAATTATTATGGGATATAAACTTGTAGTGTGGAATTTCACAGACGGTGTTTTACAGAAAGGCAGACAAGCTGTTTGTCAACGAATGTGATAAAAACCCGTATATTATATGGGTTAAGGTATGTATTTGAATGCGAAAATCAAAATTGTAATCCAAATATTGTTGGTGAACGAATATCTACAGAAAAGGCTAGTATTTATGAACAGTTATGGAGTTGCGGTTGCCATTTTGTTGAAATCACCTTCGTGATAAGAGTCGTTTTGCACCGGTACATGTGTTGGAACTTATTTAAATTCACAAGATGGTGTTCATTGTTGATGCCATTCTTTGGTAAAGGACTAGTTGTGCGTGTTTTAACAGAAAGGCAGACAAGCTGTTTGTCAACGAATGTGATAAAAACCCGTATATTATATGGGTTAAGGTATGTATTTGAATGCGAAAATCAAAATTGTAATCCAAATATTGTTGGTGAACGAATATCTACAGAAAAGGCTAGTATTTATGAACAGTTATGGAGTTGCGGTTGCCATTTTGTTGGAAATCACCTTCGTGATAAGAGTCGTTTTGCACCGGTACATGTGTTGGAACTTATTTAAATTCACAAGATGGTGTTCATTGTTGATGCCATTCTTTGGTAAAGGACTAGTTGTGCGT"

SARS_COV_PROTEIN_DEFAULT = "MFLFLLSLTSTGSEVGKVNGLTSVSLPSGFQPLVDYQTFTSDYLQPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATVSIDIQAYEQAVKIDDSRFEVLADHKKQLTPLLREVGELKNVVLPLVGTALLTLTFNYNAEDFLPFYQHLVFNKFLSFHSGTVRLPNLQSMNYNPNFTGHILRQSSETRIDLLFLKVNDRVSDIDLFCKFLTLRNIEFVLADHKKQLTPLLREVGELKNVVLPLVGTALLTLTFNYNAEDFLPFYQHLVFNKFLSFHSGTVRLPNLQSMNYNPNFTGHILRQSSETRIDLLFLKVNDR"
SARS_COV2_PROTEIN_DEFAULT = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATVSIDLFCKFLTLRNIEFVLADHKKQLTPLLREVGELKNVVLPLVGTALLTLTFNYNAEDFLPFYQHLVFNKFLSFHSGTVRLPNLQSMNYNPNFTGHILRQSSETRIDLLFLKVNDRVSIDLFCKFLTLRNIEFVLADHKKQLTPLLREVGELKNVVLPLVGTALLTLTFNYNAEDFLPFYQHLVFNKFLSFHSGTVRLPNLQSMNYNPNFTGHILRQSSETRIDLLFLKVNDRVSIDLFCKFLTLRNIEFVLADHKKQLTPLLREVGELKNVVLPLVGTALLTLTFNYNAEDFLPFYQHLVFNKFLSFHSGTVRLPNLQSMNYNPNFTGHILRQSSETRIDLLFLKVNDRV"

# Input widgets
sequence1_input = widgets.Text(description='Sequence 1:', value=SARS_COV_DNA_DEFAULT, layout=widgets.Layout(width='auto'))
sequence2_input = widgets.Text(description='Sequence 2:', value=SARS_COV2_DNA_DEFAULT, layout=widgets.Layout(width='auto'))

algorithm_selector = widgets.Dropdown(
    options=[('Needleman-Wunsch (Global)', 'needleman_wunsch'), ('Smith-Waterman (Local)', 'smith_waterman')],
    value='needleman_wunsch',
    description='Algorithm:'
)

mode_selector = widgets.RadioButtons(
    options=['DNA', 'Protein'],
    value='DNA',
    description='Mode:'
)

# New widgets for penalties
match_reward_input = widgets.IntText(description='DNA Match:', value=1, min=-10, max=10)
mismatch_penalty_input = widgets.FloatText(description='DNA Mismatch:', value=-1.0, min=-10.0, max=0.0, step=0.1)
# Changed to FloatText for decimal gap penalties. Gap penalties are typically <= 0.
gap_penalty_input = widgets.FloatText(description='Gap Penalty:', value=-3.0, min=-10.0, max=0.0, step=0.1)

# New widget for protein scoring matrix selection
protein_matrix_selector = widgets.Dropdown(
    options=list(PROTEIN_MATRICES.keys()),
    value='BLOSUM62',
    description='Protein Matrix:'
)

analyze_discrepancies_checkbox = widgets.Checkbox(
    value=False,
    description='Analyze DNA/Protein Discrepancies',
    disabled=False,
    indent=False
)

run_button = widgets.Button(description='Run Alignment')
output_area = widgets.Output()

def update_sequences_based_on_mode(change):
    if change.new == 'DNA':
        sequence1_input.value = SARS_COV_DNA_DEFAULT
        sequence2_input.value = SARS_COV2_DNA_DEFAULT
        match_reward_input.disabled = False
        mismatch_penalty_input.disabled = False
        protein_matrix_selector.disabled = True
        # Re-run for new defaults and settings
        execute_alignment_analysis()
    elif change.new == 'Protein':
        sequence1_input.value = SARS_COV_PROTEIN_DEFAULT
        sequence2_input.value = SARS_COV2_PROTEIN_DEFAULT
        match_reward_input.disabled = True
        mismatch_penalty_input.disabled = True
        protein_matrix_selector.disabled = False
        # Re-run for new defaults and settings
        execute_alignment_analysis()

mode_selector.observe(update_sequences_based_on_mode, names='value')

# Initialize disabled state for protein-specific widgets if DNA is default
if mode_selector.value == 'DNA':
    protein_matrix_selector.disabled = True
else:
    match_reward_input.disabled = True
    mismatch_penalty_input.disabled = True

def execute_alignment_analysis(change=None):
    with output_area:
        output_area.clear_output()

        s1 = sequence1_input.value.upper()
        s2 = sequence2_input.value.upper()
        selected_algorithm = algorithm_selector.value
        selected_mode = mode_selector.value

        dna_match = match_reward_input.value
        dna_mismatch = mismatch_penalty_input.value
        gap_penalty = gap_penalty_input.value
        protein_matrix_name = protein_matrix_selector.value

        try:
            if analyze_discrepancies_checkbox.value:
                # Force global DNA alignment
                print("--- Performing Global DNA Alignment for Discrepancy Analysis ---")
                dna_score, dna_aligned1, dna_aligned2 = needleman_wunsch(
                    SARS_COV_DNA_DEFAULT,
                    SARS_COV2_DNA_DEFAULT,
                    lambda c1, c2: get_score(c1, c2, mode='DNA', dna_match=2, dna_mismatch=-1),
                    gap_penalty=-2
                )
                print(f"DNA Alignment Score: {dna_score}")
                print("Aligned DNA Sequences (partial display):")
                color_align(dna_aligned1[:100], dna_aligned2[:100]) # Display a segment
                dna_stats = alignment_stats(dna_aligned1, dna_aligned2, mode='DNA')
                print(f"DNA Identity: {dna_stats['identity']}% | Total Gaps: {dna_stats['total_gaps']} | Gap Blocks: {dna_stats['num_gap_blocks']} | Avg Gap Length: {dna_stats['avg_gap_block_length']} | Frameshift Indels: {dna_stats['frameshift_indel_count']}")

                # Force global Protein alignment
                print("\n--- Performing Global Protein Alignment for Discrepancy Analysis ---")
                print(f"Using Protein Scoring Matrix: {protein_matrix_name}")
                protein_score, protein_aligned1, protein_aligned2 = needleman_wunsch(
                    SARS_COV_PROTEIN_DEFAULT,
                    SARS_COV2_PROTEIN_DEFAULT,
                    lambda c1, c2: get_score(c1, c2, mode='Protein', protein_scoring_matrix_name=protein_matrix_name),
                    gap_penalty=-10 # Protein gap penalties are usually higher
                )
                print(f"Protein Alignment Score: {protein_score}")
                print("Aligned Protein Sequences (partial display):")
                color_align(protein_aligned1[:100], protein_aligned2[:100]) # Display a segment
                protein_stats = alignment_stats(protein_aligned1, protein_aligned2, mode='Protein', protein_scoring_matrix_name=protein_matrix_name)
                print(f"Protein Identity: {protein_stats['identity']}% | Total Gaps: {protein_stats['total_gaps']} | Gap Blocks: {protein_stats['num_gap_blocks']} | Avg Gap Length: {protein_stats['avg_gap_block_length']}")

                print("\n--- Analyzing Discrepancies between DNA and Protein Alignments ---")
                discrepancy_results = analyze_alignment_discrepancies(
                    dna_aligned1,
                    dna_aligned2,
                    protein_aligned1,
                    protein_aligned2
                )

                print("\nSynonymous Mutations (DNA change, Protein same):")
                if discrepancy_results['synonymous_mutations']:
                    for m in discrepancy_results['synonymous_mutations']:
                        print(f"  Protein Pos {m['protein_position']}: {m['dna1_codon']} -> {m['dna2_codon']} (AA: {m['aa1']} -> {m['aa2']})")
                else:
                    print("  None found in this alignment.")

                print("\nNon-Synonymous Mutations (DNA change, Protein change):")
                if discrepancy_results['non_synonymous_mutations']:
                    for m in discrepancy_results['non_synonymous_mutations']:
                        print(f"  Protein Pos {m['protein_position']}: {m['dna1_codon']} -> {m['dna2_codon']} (AA: {m['aa1']} -> {m['aa2']})")
                else:
                    print("  None found in this alignment.")

                print("\nDNA segment differences where translation was unclear (e.g., gaps or incomplete codons):")
                if discrepancy_results['dna_gaps_not_full_codons']:
                    for m in discrepancy_results['dna_gaps_not_full_codons']:
                        print(f"  Protein Pos {m['protein_position']}: DNA {m['dna1_segment']} vs {m['dna2_segment']} (Protein: {m['protein_segment1']} vs {m['protein_segment2']})")
                else:
                    print("  None found in this alignment.")

                print("\n--- Interpretation ---")
                print("The differences in DNA alignment that result in the same amino acid in the protein alignment are synonymous mutations. These mutations do not change the protein sequence and are often considered 'silent'.")
                print("Differences in DNA that lead to different amino acids are non-synonymous mutations. These changes alter the protein and can have functional consequences.")
                print("Differences in DNA segments that couldn't be clearly translated to protein (often due to alignment gaps leading to incomplete codons) indicate insertions or deletions at the DNA level that affect the reading frame or length, which are highly significant at the protein level.")

            else:
                # Original logic for single alignment based on mode_selector
                score_fn_with_mode = lambda char1, char2: get_score(
                    char1, char2,
                    mode=selected_mode,
                    dna_match=dna_match,
                    dna_mismatch=dna_mismatch,
                    protein_scoring_matrix_name=protein_matrix_name
                )

                score = 0
                a1 = ""
                a2 = ""

                if selected_algorithm == 'needleman_wunsch':
                    score, a1, a2 = needleman_wunsch(s1, s2, score_fn_with_mode, gap_penalty=gap_penalty)
                elif selected_algorithm == 'smith_waterman':
                    score, a1, a2 = smith_waterman(s1, s2, score_fn_with_mode, gap_penalty=gap_penalty)

                print(f"\nAlignment Score: {score}")
                if selected_mode == 'Protein':
                    print(f"Using Protein Scoring Matrix: {protein_matrix_name}")
                print("\nAligned Sequences:")
                color_align(a1, a2)

                stats = alignment_stats(a1, a2, mode=selected_mode, protein_scoring_matrix_name=protein_matrix_name)
                print(f"Identity: {stats['identity']}% | Total Gaps: {stats['total_gaps']} | Gap Blocks: {stats['num_gap_blocks']} | Avg Gap Length: {stats['avg_gap_block_length']}" + (f" | Frameshift Indels: {stats['frameshift_indel_count']}" if selected_mode == 'DNA' else ""))

        except Exception as e:
            print(f"An error occurred: {e}")
            print("Please ensure sequences are valid and functions are correctly defined.")


run_button.on_click(execute_alignment_analysis)

# Group widgets for display
control_panel = widgets.VBox([
    sequence1_input,
    sequence2_input,
    algorithm_selector,
    mode_selector,
    widgets.HBox([match_reward_input, mismatch_penalty_input]), # Group DNA penalties
    gap_penalty_input,
    protein_matrix_selector,
    analyze_discrepancies_checkbox, # Add the new checkbox
    run_button
])

display(widgets.VBox([control_panel, output_area]))